In [6]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

## Import and Split Data

In [7]:
# 1. Create temporal splits
df = pd.read_csv('data/fr_en_features_train.csv')

/var/folders/kr/jrpfj1251_vd5r5mh8ydlh940000gn/T/ipykernel_23962/2203866952.py:2: DtypeWarning: Columns (0: Reflex) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/fr_en_features_train.csv')


In [8]:
# Temporal train/test split.
# Each (user_id, sent_id) is one contiguous block of token rows (one practice instance).
# For every user, repeatedly peel off their *last-seen* sentence (in file/appearance
# order) into the test set, then their next-to-last, etc., until the test set reaches
# 25% of all rows -- but never take more than half of any single user's sentences.
# If the half-per-user cap is hit before reaching 25%, we stop there (25% is the max
# target, not a guarantee).
df = df.reset_index(drop=True)
total_rows = len(df)
target_test = 0.25 * total_rows

# Per sentence block: rank from the end (0 = last seen) and a per-user cap of half
# the user's sentences (floor, so we never exceed half).
blocks = df[['user_id', 'sent_id']].drop_duplicates()
blocks['appear'] = blocks.groupby('user_id').cumcount()
n_sents = blocks.groupby('user_id')['appear'].transform('size')
blocks['from_end'] = n_sents - 1 - blocks['appear']
blocks['cap'] = n_sents // 2
blocks = blocks.set_index(['user_id', 'sent_id'])

idx = df.set_index(['user_id', 'sent_id']).index
from_end = pd.Series(idx.map(blocks['from_end']), index=df.index)
cap = pd.Series(idx.map(blocks['cap']), index=df.index)

# Peel one round (last sentence of every eligible user) at a time until we hit 25%
# or every user has reached their half cap.
test_mask = pd.Series(False, index=df.index)
rank = 0
max_cap = int(blocks['cap'].max())
while test_mask.sum() < target_test and rank < max_cap:
    test_mask |= (from_end == rank) & (from_end < cap)
    rank += 1

test_df = df[test_mask].reset_index(drop=True)
train_df = df[~test_mask].reset_index(drop=True)

print(f"train: {len(train_df)} rows ({len(train_df)/total_rows:.2%})")
print(f"test:  {len(test_df)} rows ({len(test_df)/total_rows:.2%})")
print(f"reached 25% target: {len(test_df) >= target_test}")


train: 656753 rows (74.99%)
test:  218982 rows (25.01%)
reached 25% target: True


## Separate features (X) and labels (y)

In [ ]:
# Complete - all features 
X_train = train_df.drop(columns=['p_recall'])
y_train = train_df['p_recall']

X_test = test_df.drop(columns=['p_recall'])
y_test = test_df['p_recall']

In [12]:
# Baseline - w/o linguistic features
bl_train_df = train_df.drop(
    columns = ["POS","Dependency-Relation","Dependancy-Head","syllable_count","ortho_freq","lev_ratio","Definite","Gender","Number","Person","PronType","Mood","Tense","VerbForm","Reflex"]
    )

bl_test_df = test_df.drop(
    columns= ["POS","Dependency-Relation","Dependancy-Head","syllable_count","ortho_freq","lev_ratio","Definite","Gender","Number","Person","PronType","Mood","Tense","VerbForm","Reflex"]
)

X_bl_train = train_df.drop(columns=['p_recall'])
y_bl_train = train_df['p_recall']

X_bl_test = test_df.drop(columns=['p_recall'])
y_bl_test = test_df['p_recall']

In [13]:
# Isolated - linguistic only
ling_train_df = train_df.drop(
    columns=["time","nth_occurrence"]
)

ling_test_df = test_df.drop(
    columns = ["time","nth_occurrence"]
)

X_ling_train = train_df.drop(columns=['p_recall'])
y_ling_train = train_df['p_recall']

X_ling_test = test_df.drop(columns=['p_recall'])
y_ling_test = test_df['p_recall']

## Train

In [ ]:
# all parameters not specified are set to their defaults
complete_logRegr = LogisticRegression()
bl_logRegr = LogisticRegression()
ling_logRegr = LogisticRegression()

complete_logRegr.fit(X_train, y_train)
bl_logRegr.fit(X_bl_train, y_bl_train)
ling_logRegr.fit(X_ling_train, y_ling_train)